Build the RISCV target

In [29]:
import tvm
import tvm.relax as relax
from tvm import meta_schedule as ms
from tvm.script import relax as R
from tvm.script import tir as T

target = tvm.target.Target(
    "llvm -mtriple=riscv64-unknown-linux-gnu "
    "-mattr=+64bit,+m,+a,+f,+d,+c,+v "
    "-mcpu=generic-rv64 "
    "-mabi=lp64d "
)

Register intrinsics

In [30]:
from tvm.tir.tensor_intrin.riscv_cpu import (
    register_riscv_intrinsics,
    get_max_elems,
)
from tvm.runtime import DataType
from tvm.target.codegen import llvm_get_vector_width

BATCH, CHANNELS, H, W = 14, 23, 67, 99
SHAPE = (BATCH, CHANNELS)
DTYPE = "float32"

In [31]:
@tvm.script.ir_module
class AddModule:
    @R.function
    def main(x: R.Tensor(SHAPE, DTYPE), y: R.Tensor(SHAPE, DTYPE)):
        with R.dataflow():
            gv = relax.op.add(x, y)
            R.output(gv)
        return gv

In [32]:
with target:
    inventory = register_riscv_intrinsics(target)

vlen      = llvm_get_vector_width(target)
dt        = DataType(DTYPE)
max_elems = get_max_elems(vlen, lmul=8, sew=dt.bits)

n_elems   = min(SHAPE[0], max_elems)

intrin_name = "rvv_add_32u8_4x32i8_4i32"
intrin = tvm.tir.TensorIntrin.get(intrin_name)

In [33]:
import tvm.te as te
import tvm.tir as tir

#mod = tvm.IRModule({"main": intrin.impl})
mod = AddModule

with tvm.transform.PassContext(opt_level=3):
    mod = LegalizeOps()(AddModule)

mod.show()

In [34]:
@tvm.transform.module_pass(opt_level=3)
def ApplyTensorize(mod, ctx):
    for gv, func in mod.functions.items():
        if isinstance(func, tvm.tir.PrimFunc):
            sch = tvm.tir.Schedule(func)
            try:
                block = sch.get_block("T_add")
                outer, inner = sch.get_loops(block)
                sch.tensorize(outer, intrin_name)
                mod[gv] = sch.mod["main"]
            except Exception as e:
                print(f"Tensorize skipped for {gv}: {e}")
    return mod

In [35]:
mod = ApplyTensorize(mod)

Tensorize skipped for I.GlobalVar("add"): ScheduleError: An error occurred in the schedule primitive 'tensorize'.

The IR with diagnostic is:
# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func(private=True)
    def main(var_x: T.handle, var_y: T.handle, var_T_add: T.handle):
        T.func_attr({"tir.noalias": True})
        x = T.match_buffer(var_x, (T.int64(14), T.int64(23)))
        y = T.match_buffer(var_y, (T.int64(14), T.int64(23)))
        T_add = T.match_buffer(var_T_add, (T.int64(14), T.int64(23)))
        with T.block("root"):
            T.reads()
            T.writes()
            for ax0 in range(T.int64(14)):
                for ax1 in range(T.int64(23)):
                    with T.block("T_add"):
                        v_ax0 = T.axis.spatial(T.int64(14), ax0)
                        v_ax1 = T.axis.spatial(T.int64(23), ax1)
                        T.reads(x[v_ax0, v_ax1], y[v_ax0, v_ax1])
                      

In [36]:
ex = tvm.compile(mod, target=target)

In [37]:
import os
OUTPUT_DIR = "output/intrinsics"

ex = tvm.compile(mod, target=target)

so_path = os.path.join(OUTPUT_DIR, f"intrinsic_add.so")
asm_path = os.path.join(OUTPUT_DIR, f"intrinsic_add.asm")

ex.export_library(
    so_path,
    cc="riscv64-linux-gnu-gcc"
)
print(f"    [INFO] Shared lib -> {so_path}")

import subprocess
result = subprocess.run(
    ["llvm-objdump-18", "-d", "--mattr=+v", so_path],
    capture_output=True, text=True, check=True
)
with open(asm_path, "w") as f:
    f.write(result.stdout)
print(f"    [INFO] Disassembly -> {asm_path}")

    [INFO] Shared lib -> output/intrinsics/intrinsic_add.so
    [INFO] Disassembly -> output/intrinsics/intrinsic_add.asm
